## Image download Notebook
#### Notebook for downloading the image data and metadata directly from the ISIC archive.

In [1]:
import csv
import os
import random
import requests

#Here we define a python function (download_isic_images) which searches for images in the ISIC archive. 
def download_isic_images(diagnosis, limit, save_dir, records, pool_multiplier=3):
    os.makedirs(save_dir, exist_ok=True)
    url = "https://api.isic-archive.com/api/v2/images/search/"
    params = {
        "query": f'diagnosis_1:"{diagnosis}" AND image_type:"dermoscopic"',
        "limit": limit * pool_multiplier
    }

    response = requests.get(url, params=params)
    response.raise_for_status()
    results = response.json().get("results", [])

    #Shuffles the order of the images and selects the desired amount per class. 
    random.shuffle(results)
    selected_results = results[:limit]

    #Loops over selected_results
    #Downloads the image files and saves the image information.
    for i, item in enumerate(selected_results, 1):
        isic_id = item["isic_id"]
        img_url = item["files"]["full"]["url"]

        img_data = requests.get(img_url)
        img_data.raise_for_status()

        filename = f"{isic_id}_{diagnosis.lower()}_dermoscopic.jpg"
        file_path = os.path.join(save_dir, filename)

        with open(file_path, "wb") as f:
            f.write(img_data.content)

        records.append({
            "filename": filename,
            "file_path": file_path,
            "diagnosis": diagnosis,
            "image_type": "dermoscopic"
        })
        print(f"[{i}/{len(selected_results)}] Laddade ner {file_path}")

all_records = []

#Calls the python function (download_isic_images).
#The images are saved in the map "data"
download_isic_images("Benign", 50, "data", all_records)
download_isic_images("Malignant", 50, "data", all_records)

#Shuffles the line order of the classes in the CSV-file.
random.shuffle(all_records)

#The image information is saved in a table-file "labels.csv". 
csv_path = "data/labels.csv"
with open(csv_path, "w", newline="", encoding="utf-8") as csvfile:
    fieldnames = ["filename", "file_path", "diagnosis", "image_type"]
    writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(all_records)

[1/50] Laddade ner data/ISIC_0000052_benign_dermoscopic.jpg
[2/50] Laddade ner data/ISIC_0000098_benign_dermoscopic.jpg
[3/50] Laddade ner data/ISIC_0000008_benign_dermoscopic.jpg
[4/50] Laddade ner data/ISIC_0000087_benign_dermoscopic.jpg
[5/50] Laddade ner data/ISIC_0000039_benign_dermoscopic.jpg
[6/50] Laddade ner data/ISIC_0000066_benign_dermoscopic.jpg
[7/50] Laddade ner data/ISIC_0000099_benign_dermoscopic.jpg
[8/50] Laddade ner data/ISIC_0000005_benign_dermoscopic.jpg
[9/50] Laddade ner data/ISIC_0000084_benign_dermoscopic.jpg
[10/50] Laddade ner data/ISIC_0000119_benign_dermoscopic.jpg
[11/50] Laddade ner data/ISIC_0000062_benign_dermoscopic.jpg
[12/50] Laddade ner data/ISIC_0000016_benign_dermoscopic.jpg
[13/50] Laddade ner data/ISIC_0000028_benign_dermoscopic.jpg
[14/50] Laddade ner data/ISIC_0000093_benign_dermoscopic.jpg
[15/50] Laddade ner data/ISIC_0000045_benign_dermoscopic.jpg
[16/50] Laddade ner data/ISIC_0000010_benign_dermoscopic.jpg
[17/50] Laddade ner data/ISIC_000